# 02 – Data cleaning and exploratory analysis

In [ ]:
import os
import re

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from sklearn.feature_extraction.text import CountVectorizer

df = pd.read_csv("data/books.csv")
print(df.shape)
df.head()

## 1. Structure and basic checks

Check data types, missing values and duplicates. The UPC is unique per book, so it is the
natural key for duplicates.

In [ ]:
df.info()

In [ ]:
print("Missing values:", df.isna().sum().sum())
print("Duplicate UPCs:", df["upc"].duplicated().sum())
print("Duplicate titles:", df["title"].duplicated().sum())

## 2. Convert raw text columns to numbers

The scraper stored values exactly as shown on the site. They are converted here:
- `price`: `"£50.10"` → `50.10`
- `rating`: `"Three"` → `3`
- `availability`: `"In stock (20 available)"` → `stock = 20`

In [ ]:
RATINGS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

df["price"] = df["price"].str.replace("£", "").astype(float)
df["rating"] = df["rating"].map(RATINGS)
# grab the number out of "In stock (20 available)"
df["stock"] = df["availability"].str.extract(r"(\d+)")[0].astype(int)
df = df.drop(columns="availability")

df[["price", "rating", "stock"]].describe()

## 3. Clean the descriptions

Most descriptions contain the site's short teaser followed by the full text, and end with
`" ...more"`. Example: `"...sans vraie révolution, s Dans une France assez proche..."`.
The duplicated teaser would count the same words twice, so it is removed.

In [ ]:
def clean_description(text):
    # Remove the "...more" link text at the end
    text = text.removesuffix("...more").strip()

    # Both the teaser and the full text start with the same words.
    # Look for the first 40 characters again, starting after position 0.
    start = text.find(text[:40], 1)

    # If found, the full text starts there, so drop the teaser before it
    if start > 0:
        text = text[start:]
    return text


print("Ending with '...more':", df["description"].str.endswith("...more").sum())
before = df["description"].str.len()
df["description"] = df["description"].apply(clean_description)
# a text counts as fixed if it got shorter by more than just the "...more" bit
print("Repeated teasers removed:", (df["description"].str.len() < before - 10).sum())
print("Average length before/after:", round(before.mean()), "/", round(df["description"].str.len().mean()))

## 4. Find non-English descriptions

A simple check: the share of words that are very common English words. Normal English text
scores around 0.25, a non-English text scores close to 0.

In [ ]:
COMMON_EN = {"the", "and", "of", "to", "a", "in", "is", "that", "his", "her", "with", "for"}

def english_ratio(text):
    words = re.findall(r"[a-z]+", text.lower())
    # max(..., 1) so we never divide by zero on an empty text
    return sum(w in COMMON_EN for w in words) / max(len(words), 1)

df["en_ratio"] = df["description"].apply(english_ratio)
df.sort_values("en_ratio")[["title", "category", "en_ratio", "description"]].head(4)

Only *Soumission* (French) is not English. The *Kindle Paperwhite User's Guide* also scores 0,
but only because its description is just the title, so it stays.
*Soumission* is removed: its French words would never match the English vocabulary of the other books.

In [ ]:
# reset_index renumbers the rows 0, 1, 2... after removing one
df = df[df["title"] != "Soumission"].drop(columns="en_ratio").reset_index(drop=True)
print(df.shape)

## 5. Check the images

Every image must open without errors (a broken file would raise an error here).
Sizes differ, so they will be resized to a common size in the feature engineering step.

In [ ]:
widths, heights = [], []
for path in df["image_path"]:
    with Image.open(path) as img:
        widths.append(img.width)
        heights.append(img.height)

print("Images opened:", len(widths))
print(f"Width: {min(widths)}–{max(widths)} px, height: {min(heights)}–{max(heights)} px")

## 6. Target label

Fiction = 1, Nonfiction = 0. The classes are imbalanced, so always guessing "Nonfiction" is the
baseline the model needs to beat.

In [ ]:
df["is_fiction"] = (df["category"] == "Fiction").astype(int)

counts = df["category"].value_counts()
print(counts)
print(f"Majority baseline accuracy: {counts.max() / counts.sum():.1%}")

## 7. Exploratory analysis

The analysis uses a copy (`eda`), so the extra columns made here are not saved with the cleaned data.

### Numeric columns by category
The site states that prices and ratings were randomly assigned, so we expect no real difference.

In [ ]:
eda = df.copy()
eda["desc_length"] = eda["description"].str.split().str.len()
eda["title_length"] = eda["title"].str.split().str.len()

eda.groupby("category")[["price", "rating", "stock", "desc_length", "title_length"]].mean().round(2)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, col in zip(axes, ["price", "rating", "stock", "desc_length"]):
    eda.boxplot(column=col, by="category", ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
fig.suptitle("Numeric features by category")
plt.tight_layout()
os.makedirs("figures", exist_ok=True)
plt.savefig("figures/numeric_by_category.png", dpi=150)
plt.show()

### Title patterns
Nonfiction titles often have a subtitle after a colon ("Sapiens: A Brief History of Humankind"),
while fiction titles often belong to a series ("Private Paris (Private #10)").

In [ ]:
eda["title_has_colon"] = eda["title"].str.contains(":")
eda["title_has_series"] = eda["title"].str.contains("#")

eda.groupby("category")[["title_has_colon", "title_has_series"]].mean().round(2)

### Most common words in the descriptions per category
English stop words ("the", "and", ...) are removed, since they appear everywhere.

In [ ]:
vectorizer = CountVectorizer(stop_words="english")

for category in ["Fiction", "Nonfiction"]:
    texts = df.loc[df["category"] == category, "description"]
    # count each word per book, then add the counts up over all books in this category
    word_counts = vectorizer.fit_transform(texts).toarray().sum(axis=0)
    words = pd.Series(word_counts, index=vectorizer.get_feature_names_out())
    print(category + ":", ", ".join(words.nlargest(15).index))

## 8. Save the cleaned data

Engineered features are created in the next notebook.

In [ ]:
df.to_csv("data/books_clean.csv", index=False, encoding="utf-8")
print(df.shape)
df.head()